# D-Flow on Circles

Reverse sampling with a pretrained flow-matching UNet on the **n=4 circles** dataset.

Given an observed image $y$, we find the optimal latent $x_0^*$ such that

$$x_0^* = \arg\min_{x_0} \mathcal{L}(x_0), \quad \mathcal{L}(x_0) = \|G(x_0) - y\|^2$$

where $G(x_0) = x_1$ is the forward ODE flow. We then resample from $x_0^*$ to verify recovery.

## Setup

**First time on Colab — do this once:**

1. Upload the following to a folder in Google Drive (e.g. `My Drive/d-flow/`):
   - `models/` directory (entire folder)
   - `sampler/` directory (entire folder)
   - `outputs/fm_unet_n4/checkpoints/epoch_0150.pt`
   - `data/circles-64-n4.pt`

2. Run the cell below — it will mount Drive and install `cmocean`.

Set `DRIVE_PROJECT_DIR` to match wherever you put the files.

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ

if IN_COLAB:
    # ── install extra deps (torch is pre-installed on Colab) ──────────────────
    import subprocess
    subprocess.run(["pip", "install", "cmocean", "-q"], check=True)

    # ── mount Google Drive ────────────────────────────────────────────────────
    from google.colab import drive
    drive.mount("/content/drive")

    # ── point to your uploaded project folder ─────────────────────────────────
    DRIVE_PROJECT_DIR = "/content/drive/MyDrive/d-flow"   # ← adjust if needed
    sys.path.insert(0, DRIVE_PROJECT_DIR)

    CKPT_PATH = f"{DRIVE_PROJECT_DIR}/outputs/fm_unet_n4/checkpoints/epoch_0150.pt"
    DATA_PATH = f"{DRIVE_PROJECT_DIR}/data/circles-64-n4.pt"

else:
    # ── local machine ─────────────────────────────────────────────────────────
    sys.path.insert(0, "..")
    CKPT_PATH = "../outputs/fm_unet_n4/checkpoints/epoch_0150.pt"
    DATA_PATH = "../data/circles-64-n4.pt"

print("Environment:", "Colab" if IN_COLAB else "Local")
print("Checkpoint :", CKPT_PATH)
print("Data       :", DATA_PATH)

In [ ]:
import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import cmocean

from models.unet.default import Unet
from sampler.ode_samplers import ode_integrate

matplotlib.rcParams["font.family"] = "TeX Gyre Pagella"
matplotlib.rcParams["mathtext.fontset"] = "custom"
matplotlib.rcParams["mathtext.rm"] = "TeX Gyre Pagella"
CMAP = cmocean.cm.dense_r

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1 — Load pretrained model

In [ ]:
model = Unet(ch=24, ch_mul=[1, 2, 2], att_channels=[0, 1, 1], groups=8, dropout=0.0)

# Checkpoint was saved from AveragedModel(torch.compile(model)):
# keys have 'module._orig_mod.' prefix — strip both wrappers
ckpt = torch.load(CKPT_PATH, weights_only=True, map_location="cpu")
state_dict = {
    k.replace("module._orig_mod.", ""): v
    for k, v in ckpt.items()
    if k != "n_averaged"
}
model.load_state_dict(state_dict)
model = model.to(DEVICE).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded  n_averaged={ckpt['n_averaged'].item():,}  |  params={n_params:,}")

## 2 — Pick observation $y$

In [ ]:
data = torch.load(DATA_PATH, weights_only=True)
test = data["test"]          # (1000, 1, 64, 64)

IDX = 0                      # change to try different samples
y   = test[IDX : IDX + 1].to(DEVICE)

fig, ax = plt.subplots(1, 1, figsize=(3, 3))
ax.imshow(y[0, 0].cpu(), cmap=CMAP, vmin=0, vmax=1, interpolation="nearest")
ax.set_title(r"Observation $y$", fontsize=11)
ax.axis("off")
plt.tight_layout()
plt.show()

## 3 — Forward ODE helper

Wraps the UNet into the `ode_integrate` interface. The model expects $t \in [0,1]$ (it internally scales by 1000).

In [ ]:
def ode_func(t, x):
    """Velocity model: scalar t (0-dim tensor), x shape (B,1,64,64)."""
    t_batch = t.expand(x.shape[0])
    return model(x, t_batch)

def forward_ode(x0, n_steps=100, sampler="rk4"):
    """Integrate x0 → x1 with the pretrained velocity field."""
    return ode_integrate(
        ode_func=ode_func,
        init_x=x0,
        ode_opts={"options": {"step_size": 1.0 / n_steps}},
        sampler=sampler,
    ).clamp(0.0, 1.0)

## 4 — D-Flow: optimise $x_0$ to minimise $\mathcal{L}(x_0) = \|G(x_0) - y\|^2$

We treat $x_0 \in \mathbb{R}^{1 \times 64 \times 64}$ as the free variable and run L-BFGS.

For the optimisation we use **Euler with 20 steps** — this keeps the computation graph shallow enough to backprop through efficiently. The final resample uses RK4 with 100 steps for accuracy.

In [ ]:
torch.manual_seed(0)
x0 = torch.randn_like(y, requires_grad=True)

optimizer = torch.optim.LBFGS(
    [x0], lr=1.0, max_iter=20, line_search_fn="strong_wolfe"
)

# Use Euler with 20 steps for the optimisation — shallow graph, fast backprop
OPT_STEPS  = 30
OPT_N_STEPS = 20
costs = []

for step in range(OPT_STEPS):
    def closure():
        optimizer.zero_grad()
        x1 = forward_ode(x0, n_steps=OPT_N_STEPS, sampler="euler")
        L  = torch.mean((x1 - y) ** 2)
        L.backward()
        costs.append(L.item())
        return L
    optimizer.step(closure)
    if (step + 1) % 5 == 0:
        print(f"step {step+1:3d} / {OPT_STEPS}  |  L = {costs[-1]:.6f}")

x0_opt = x0.detach().clone()
print(f"\nFinal cost: {costs[-1]:.6f}")

## 5 — Optimisation cost curve

In [ ]:
c_line = cmocean.cm.dense_r(0.7)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, yscale in [(ax1, "linear"), (ax2, "log")]:
    ax.plot(costs, color=c_line, lw=1.8)
    ax.set_yscale(yscale)
    ax.set_xlabel("L-BFGS inner iteration")
    ax.set_ylabel(r"$\mathcal{L}(x_0) = \|G(x_0)-y\|^2$")
    ax.set_title(f"Cost ({yscale} scale)")
fig.suptitle("D-Flow optimisation", fontsize=11)
fig.tight_layout()
plt.show()

## 6 — Resample from $x_0^*$ and compare to $y$

Run the forward ODE from the optimised latent $x_0^*$ and verify we recover the observation.

In [ ]:
with torch.no_grad():
    x1_recovered = forward_ode(x0_opt)

mse = torch.mean((x1_recovered - y) ** 2).item()
print(f"MSE(x1_recovered, y) = {mse:.6f}")

# ── plot ──────────────────────────────────────────────────────────────────────
imgs = {
    r"Observation $y$":              y[0, 0].cpu(),
    r"Optimised $x_0^*$":            x0_opt[0, 0].cpu(),
    r"Resampled $G(x_0^*)$":         x1_recovered[0, 0].cpu(),
    r"Residual $|G(x_0^*) - y|$":   (x1_recovered - y).abs()[0, 0].cpu(),
}

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5),
                         gridspec_kw={"wspace": 0.06})
for ax, (title, img) in zip(axes, imgs.items()):
    if "Residual" in title:
        im = ax.imshow(img, cmap="Reds", vmin=0, vmax=0.5, interpolation="nearest")
    else:
        im = ax.imshow(img, cmap=CMAP, vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

fig.suptitle(rf"D-Flow recovery  —  MSE = {mse:.4f}", fontsize=11)
plt.show()